# (노트) 순환신경망

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [빅데이터분석]

### Import 

In [1]:
from fastai.text.all import *
import torch 
import numpy as np

### 실제활용예시

In [2]:
path = untar_data(URLs.IMDB)

In [3]:
files = get_text_files(path)
files

(#100002) [Path('/home/cgb3/.fastai/data/imdb/train/neg/7473_4.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/11965_4.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/11897_3.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/9582_4.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/2709_2.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/7593_2.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/3936_2.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/12323_4.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/576_3.txt'),Path('/home/cgb3/.fastai/data/imdb/train/neg/12303_1.txt')...]

- 경로안에 train, test, unsup이라는 폴더가 있는데, 이 폴더들내의 모든 텍스트파일을 가져오라. 

In [4]:
dls=DataBlock(
    blocks=TextBlock.from_folder(path,is_lm=True),
    get_items=get_text_files,splitter=RandomSplitter(0.1)
).dataloaders(source=path, bs=128, seq_len=80)

- `is_lm`: 문장생성을 위해서는 is_lm=True로 설정해야함.   
- splitter: train / val 으로 나눔 

In [5]:
dls.show_batch()

,text,text_
0,"xxbos xxup warning ! ! xxmaj this review may contain spoilers . xxmaj the back of the box is misleading . xxmaj it says all this crap about kids telling ghost stories , which they do , but then it implies that they will all be killed by some killer in the woods . xxmaj this does n't happen . xxmaj the stories they tell are a little interesting , specifically the one with the dog and all that licking","xxup warning ! ! xxmaj this review may contain spoilers . xxmaj the back of the box is misleading . xxmaj it says all this crap about kids telling ghost stories , which they do , but then it implies that they will all be killed by some killer in the woods . xxmaj this does n't happen . xxmaj the stories they tell are a little interesting , specifically the one with the dog and all that licking ,"
1,"can activate ca n't change much about that . \n\n xxmaj so this pretty much sums up what it is ; a failed experiment , hyped with the now very popular xxmaj doom 3 - engine . xxmaj if you want a innovative and exciting shooter , go play half - life 2 ( again ) , or the new xxmaj portal - mod i mentioned earlier . xxmaj do yourself as a xxunk a favor , ignore the hype","activate ca n't change much about that . \n\n xxmaj so this pretty much sums up what it is ; a failed experiment , hyped with the now very popular xxmaj doom 3 - engine . xxmaj if you want a innovative and exciting shooter , go play half - life 2 ( again ) , or the new xxmaj portal - mod i mentioned earlier . xxmaj do yourself as a xxunk a favor , ignore the hype and"
2,"setting , that xxup i , frankly , could not follow , which is surprising considering the acting and dialogue could have only been the product of a xxunk 's writing . xxmaj if you like xxmaj kathy xxmaj ireland , then maybe you 'd want to see this . xxmaj the movie was probably made as a vehicle to try to get her into xxmaj hollywood , but if that was its goal i would have to say that",", that xxup i , frankly , could not follow , which is surprising considering the acting and dialogue could have only been the product of a xxunk 's writing . xxmaj if you like xxmaj kathy xxmaj ireland , then maybe you 'd want to see this . xxmaj the movie was probably made as a vehicle to try to get her into xxmaj hollywood , but if that was its goal i would have to say that i"
3,"seen the previous installments , knows that there 's only one place this prequel can end . xxmaj the last few minutes of the film are heartbreaking and the film 's end credit song beautifully encapsulates the finality of xxmaj sadako 's backstory . \n\n xxmaj do n't expect too many absolute answers here though . "" ring 0 : xxmaj birthday "" maintains the mystery and ambiguity of the first two films and once again , imagination is a","the previous installments , knows that there 's only one place this prequel can end . xxmaj the last few minutes of the film are heartbreaking and the film 's end credit song beautifully encapsulates the finality of xxmaj sadako 's backstory . \n\n xxmaj do n't expect too many absolute answers here though . "" ring 0 : xxmaj birthday "" maintains the mystery and ambiguity of the first two films and once again , imagination is a required"
4,") . xxmaj if you remove the rest of the movie and just watch amitabh play around with his character , it would still be worth a watch . xxmaj but as xxmaj insp . xxmaj narsimha , xxmaj mohanlal does n't do justice to his talent . xxmaj ajay xxunk ) is extremely mundane and the only reason i think , they cast xxmaj prashant xxmaj raj in the role of xxmaj raj is because he has a striking",". xxmaj if you remove the rest of the movie and just watch amitabh play around with his character , it would still be worth a watch . xxmaj but as xxmaj insp . xxmaj narsimha , xxmaj mohanlal does n't do justice to his talent . xxmaj ajay xxunk ) is extremely mundane and the only reason i think , they cast xxmaj prashant xxmaj raj in the role of xxmaj raj is because he has a stri

- xxbox: 새로운 텍스트의 시작 
- xxmaj: 다음단어가 대문자로 시작함을 의미 (모든단어는 기본적으로 소문자라고 생각함) 
- .. 

In [6]:
lrnr = language_model_learner(dls,AWD_LSTM,metrics=accuracy).to_fp16()

- to_fp16(): 그레디언트를 대충계산한다. 

In [7]:
lrnr.fit_one_cycle(5)

epoch,train_loss,valid_loss,accuracy,time
0,4.462815,4.136233,0.283587,07:26
1,4.335924,4.033761,0.290330,07:23
2,4.293494,3.999372,0.292693,07:22
3,4.274117,3.986934,0.293585,07:21
4,4.258859,3.984887,0.293765,07:36


In [8]:
TEXT = 'I liked this movie because' 
N_WORDS= 40 
N_SENTENCES = 5
preds = [lrnr.predict(TEXT,N_WORDS,temperature=0.75) for _ in range(N_SENTENCES)]
print("\n".join(preds))

i liked this movie because i had already heard of the Academy Award for this movie and i loved it . It was truly one of the best movies i have ever seen . i was pretty happy with it . i
i liked this movie because i wanted to be a fan of B 6.5 movies in the early 80 's and lived for a long time . It was so interesting that i was hoping for it to make it it 's "
i liked this movie because it was a fun movie . i thought it was a cheap , gritty movie about a man who has been around for years , his father is a film maker , i believe he picked out the Dutch
i liked this movie because they were watching this film on TV , but this was one of the worst movies ever made . So , i tried not to throw in this movie . The plot is simple , and this
i liked this movie because i liked Jaws when there was a plot and it was a great movie . The only flaw in this movie is that there is nothing going on that he does not even make it … a very


- 말이 안되는것 같은데 그럴듯해보이긴함 

### 문제의설계 

In [7]:
text= 'h e l l o '*100
text

'h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o

In [8]:
tokens= text.split(' ')[:-1]
tokens[:10]

['h', 'e', 'l', 'l', 'o', 'h', 'e', 'l', 'l', 'o']

`-` 바로직전문자로 다음문자를 맞춰보자. 
- hello 니까, h $\to$ e, e $\to$ l, l $\to$ l/o (?), l $\to$ o, o $\to$ h ... 
- l 다음에 올 문자가 조금 애매하다. 

`-` 마치 아래의 표에서 $X \to y$인 맵핑을 알아차려 $X$를 보고 $y$를 예측하듯이 

|X|y|
|:-:|:-:|
|1|2|
|2|4|
|3|6|
|1|2|
|2|4|
|3|6|
|...|...|

우리는 이제 아래를 예측하는 규칙을 알아차리는게 목표이다. 

|X|y|
|:-:|:-:|
|h|e|
|e|l|
|l|l/o|
|o|h|
|h|e|
|...|...|

### Embedding 

`-` X,y를 설정하자.  

In [9]:
X= tokens[:(len(tokens)-1)]
y= tokens[1:len(tokens)]

In [10]:
X[0],y[0]

('h', 'e')

In [11]:
X[1],y[1]

('e', 'l')

`-` 다 좋은데 학습가능한 형태로 만들어야 하므로 문자를 숫자로 바꾸자. 

In [12]:
dic = {'h': 0, 'e': 1, 'l': 2, 'o': 3}
dic

{'h': 0, 'e': 1, 'l': 2, 'o': 3}

In [13]:
dic['h'],dic['e'],dic['l'],dic['o'] 

(0, 1, 2, 3)

In [14]:
nums = [dic[i] for i in tokens]

In [15]:
tokens[:10],nums[:10]

(['h', 'e', 'l', 'l', 'o', 'h', 'e', 'l', 'l', 'o'],
 [0, 1, 2, 2, 3, 0, 1, 2, 2, 3])

`-` (맵핑방식1) 아래와 같이 문자와 숫자를 mapping 하였다.

|문자(tokens)|숫자(nums)|
|:-:|:-:|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|...|...|

`-` (맵핑방식2) 사실 아래와 같이 맵핑하는 것이 더 좋다. 위의방식대로하면 의미가 `e=1`, `l=2`가 되는데 그렇다고해서 `l이 e보다 2배가 강한 입력`을 의미하는것은 아니므로.. 

|문자(tokens)|숫자(nums)|
|:-:|:-:|
|'h'|1,0,0,0|
|'e'|0,1,0,0|
|'l'|0,0,1,0|
|'l'|0,0,1,0|
|'o'|0,0,0,1|
|'h'|1,0,0,0|
|'e'|0,1,0,0|
|'l'|0,0,1,0|
|'l'|0,0,1,0|
|'o'|0,0,0,1|
|'h'|1,0,0,0|
|'e'|0,1,0,0|
|'l'|0,0,1,0|
|'l'|0,0,1,0|
|'o'|0,0,0,1|
|...|...|

`-` 맵핑방식2로 처리하고 싶은데, 데이터 전처리 하기가 너무 하기 어려울것 같다. 
- 그런데 이러한것은 빈번하게 일어나는 상황아닌가?.. 
- 누군가 구해놓지 않았을까?.. 
- torch.nn.Embedding 

`-` 맵핑방식1의 구현 

In [16]:
_l1 = torch.nn.Linear(in_features=1,out_features=20,bias=False)

In [17]:
_x = torch.tensor([[0.0],[1.0],[2.0],[2.0],[3.0],[0.0],[1.0],[2.0],[2.0],[3.0]])
_l1(_x)

tensor([[ 0.0000, -0.0000, -0.0000,  0.0000,  0.0000, -0.0000,  0.0000, -0.0000,
          0.0000, -0.0000,  0.0000, -0.0000,  0.0000, -0.0000,  0.0000, -0.0000,
         -0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.3161, -0.5600, -0.3293,  0.0874,  0.1445, -0.4289,  0.6062, -0.1607,
          0.3336, -0.1310,  0.6508, -0.6025,  0.2383, -0.1836,  0.9252, -0.9870,
         -0.5640,  0.9336,  0.0211,  0.7182],
        [ 0.6322, -1.1199, -0.6587,  0.1748,  0.2889, -0.8579,  1.2124, -0.3215,
          0.6671, -0.2619,  1.3016, -1.2049,  0.4765, -0.3672,  1.8505, -1.9741,
         -1.1279,  1.8671,  0.0422,  1.4364],
        [ 0.6322, -1.1199, -0.6587,  0.1748,  0.2889, -0.8579,  1.2124, -0.3215,
          0.6671, -0.2619,  1.3016, -1.2049,  0.4765, -0.3672,  1.8505, -1.9741,
         -1.1279,  1.8671,  0.0422,  1.4364],
        [ 0.9483, -1.6799, -0.9880,  0.2622,  0.4334, -1.2868,  1.8186, -0.4822,
          1.0007, -0.3929,  1.9524, -1.8074,  0.7148, -0.5508,  2.7757, -2.9611,
      

- 입력: (10,1) 
- 출력: (10,20) 

`-` 맵핑방식2의 구현 

In [18]:
e1=torch.nn.Embedding(num_embeddings=4,embedding_dim=20) 

In [19]:
_x = torch.tensor([0,1,2,3,0,1,2,3])
e1(_x)

tensor([[-8.8253e-02, -1.5453e+00, -5.8171e-01, -2.0573e+00, -2.4535e-03,
         -6.3902e-01, -9.6639e-01,  1.5336e+00, -8.3725e-01,  7.3535e-01,
         -1.3649e+00,  7.2119e-01,  6.2594e-01,  9.2904e-01,  6.7484e-01,
         -4.3243e-02, -5.3145e-01, -6.2017e-01, -5.1684e-01, -1.0643e+00],
        [ 1.4227e-01, -2.6047e-01,  2.6597e-02, -8.1533e-02,  9.1741e-01,
         -4.5371e-01, -6.7137e-01, -1.6613e-02,  8.0899e-01,  1.5540e+00,
          4.8864e-01, -2.9180e-01, -2.4515e-01,  6.5575e-01,  8.3970e-01,
          1.9640e-01, -6.6256e-01, -2.0470e+00,  1.7393e-03,  7.2544e-01],
        [-1.5153e+00, -6.6169e-01, -1.0778e+00, -1.3883e+00,  1.9315e-01,
         -5.1379e-01, -3.4998e-01, -5.7776e-01, -2.2147e-01,  8.7824e-01,
          1.7484e+00,  1.6740e+00, -8.8498e-01, -4.1644e-01, -9.2193e-01,
          1.7764e+00,  5.7562e-01, -1.2898e-01, -2.9595e-01, -1.1475e+00],
        [-1.0168e+00, -9.6321e-01, -2.0250e+00,  4.7102e-01,  2.4757e-01,
         -1.8049e-01,  5.6412e-01, 

- 입력차원: (10,1) 
- 출력차원: (10,20)

`-` 결과가 똑같다? 하지만 내부 수행과정은 약간 다르다. 

In [20]:
list(_l1.parameters())

[Parameter containing:
 tensor([[ 0.3161],
         [-0.5600],
         [-0.3293],
         [ 0.0874],
         [ 0.1445],
         [-0.4289],
         [ 0.6062],
         [-0.1607],
         [ 0.3336],
         [-0.1310],
         [ 0.6508],
         [-0.6025],
         [ 0.2383],
         [-0.1836],
         [ 0.9252],
         [-0.9870],
         [-0.5640],
         [ 0.9336],
         [ 0.0211],
         [ 0.7182]], requires_grad=True)]

- 맵핑방식1의 경우 $1\times 20$의 차원의 ${\bf W}$가 존재함. 따라서 
    - ${\bf X}$: (10,1) 
    - ${\bf W}$: (1,20) 
    - ${\bf X}{\bf W}$: (10,20)

In [21]:
list(e1.parameters())

[Parameter containing:
 tensor([[-8.8253e-02, -1.5453e+00, -5.8171e-01, -2.0573e+00, -2.4535e-03,
          -6.3902e-01, -9.6639e-01,  1.5336e+00, -8.3725e-01,  7.3535e-01,
          -1.3649e+00,  7.2119e-01,  6.2594e-01,  9.2904e-01,  6.7484e-01,
          -4.3243e-02, -5.3145e-01, -6.2017e-01, -5.1684e-01, -1.0643e+00],
         [ 1.4227e-01, -2.6047e-01,  2.6597e-02, -8.1533e-02,  9.1741e-01,
          -4.5371e-01, -6.7137e-01, -1.6613e-02,  8.0899e-01,  1.5540e+00,
           4.8864e-01, -2.9180e-01, -2.4515e-01,  6.5575e-01,  8.3970e-01,
           1.9640e-01, -6.6256e-01, -2.0470e+00,  1.7393e-03,  7.2544e-01],
         [-1.5153e+00, -6.6169e-01, -1.0778e+00, -1.3883e+00,  1.9315e-01,
          -5.1379e-01, -3.4998e-01, -5.7776e-01, -2.2147e-01,  8.7824e-01,
           1.7484e+00,  1.6740e+00, -8.8498e-01, -4.1644e-01, -9.2193e-01,
           1.7764e+00,  5.7562e-01, -1.2898e-01, -2.9595e-01, -1.1475e+00],
         [-1.0168e+00, -9.6321e-01, -2.0250e+00,  4.7102e-01,  2.4757e-01,

- 맵핑방식2의 경우 $4\times 20$의 차원의 ${\bf W}$가 존재함. 따라서 
    - ${\bf X}$: (10,1) 
    - $\tilde{{\bf X}}$: (10,4) 
    - ${\bf W}$: (4,20) 
    - $\tilde{{\bf X}}{\bf W}$: (10,20)

`-` 결국 우리가 맵핑방식2처럼 구현하고 싶다고 하더라도, 입력은 아래와 같이 넣어도 무방하다. 이후에는 알아서 파이토치가 해결해준다. 

In [22]:
_x

tensor([0, 1, 2, 3, 0, 1, 2, 3])

### 네트워크 구축 

`-` 이제 숫자화된 자료를 이용하여 다시 X,y를 선언하자. 

In [23]:
X = torch.tensor(nums[:499])
y = torch.tensor(nums[1:])

In [24]:
X[0],y[0]

(tensor(0), tensor(1))

In [25]:
X[1],y[1]

(tensor(1), tensor(2))

`-` 간단한 네트워크를 통하여 학습을 시도하자. 

In [26]:
e1=torch.nn.Embedding(num_embeddings=4,embedding_dim=20) 
l1=torch.nn.Linear(in_features=20,out_features=20)
a1=torch.nn.ReLU()
l2=torch.nn.Linear(in_features=20,out_features=4)
a2=torch.nn.Softmax()

In [27]:
X.shape,e1(X).shape

(torch.Size([499]), torch.Size([499, 20]))

In [28]:
e1(X).shape,a1(l1(e1(X))).shape

(torch.Size([499, 20]), torch.Size([499, 20]))

In [29]:
a1(l1(e1(X))).shape,l2(a1(l1(e1(X)))).shape

(torch.Size([499, 20]), torch.Size([499, 4]))

In [30]:
l2(a1(l1(e1(X))))[0]

tensor([-0.2731, -0.3494, -0.0037,  0.1286], grad_fn=<SelectBackward0>)

In [31]:
a2(l2(a1(l1(e1(X)))))[0]

<ipython-input-31-17d00ccf79de>:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  a2(l2(a1(l1(e1(X)))))[0]


tensor([0.2114, 0.1959, 0.2768, 0.3159], grad_fn=<SelectBackward0>)

- $X$의 차원이 명시되지 않아서 대충 내가 알아서 계산했다는 워닝

`-` 경고가 찝찝하여 손계산해봄 $\to$ 알아서 잘 계산했음 

In [32]:
np.exp(0.1195)/(np.exp(0.1195)+np.exp(-0.3600)+np.exp(-0.1282)+np.exp(-0.5230))

0.34180289149024373

`-` 순전파차원변화 요약 

```
torch.Size([499]) ## X with levels=4 
torch.Size([499, 20]) ## 임베딩 이후
torch.Size([499, 20]) ## 선형변환1 이후
torch.Size([499, 20]) ## 렐루 이후
torch.Size([499, 4]) ## 선형변환2 이후 
torch.Size([499, 4]) ## 소프트맥스 이후 = yhat 
```

In [33]:
net = torch.nn.Sequential(
    torch.nn.Embedding(num_embeddings=4,embedding_dim=20),
    torch.nn.Linear(in_features=20,out_features=20),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=20,out_features=4))
    #torch.nn.Softmax()

In [34]:
net(X)

tensor([[ 0.1236,  0.1050,  0.0094,  0.1671],
        [ 0.2697,  0.0366, -0.2902, -0.0519],
        [ 0.0293, -0.2960, -0.3494, -0.3689],
        ...,
        [ 0.2697,  0.0366, -0.2902, -0.0519],
        [ 0.0293, -0.2960, -0.3494, -0.3689],
        [ 0.0293, -0.2960, -0.3494, -0.3689]], grad_fn=<AddmmBackward0>)

- 순전파 

`-` 손실함수, 옵티마이저 

In [35]:
loss_fn = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(net.parameters())

In [37]:
for i in range(1000):
    ## 1 
    yhat = net(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward()
    ## 4 
    optimizer.step()
    optimizer.zero_grad()

In [38]:
dic

{'h': 0, 'e': 1, 'l': 2, 'o': 3}

In [39]:
X[:7] 

tensor([0, 1, 2, 2, 3, 0, 1])

In [40]:
net(X)[:7]

tensor([[-2.3892,  7.1199, -2.0946, -2.5325],
        [-3.2633, -3.4842,  6.0539, -3.6175],
        [-5.4399, -5.5046,  3.6418,  3.6417],
        [-5.4399, -5.5046,  3.6418,  3.6417],
        [ 6.1221, -3.2766, -2.6439, -4.1996],
        [-2.3892,  7.1199, -2.0946, -2.5325],
        [-3.2633, -3.4842,  6.0539, -3.6175]], grad_fn=<SliceBackward0>)

In [41]:
y[:7]

tensor([1, 2, 2, 3, 0, 1, 2])

- 학습이 잘 되었다. 

### net의 개선 

`-` 단어수가 바뀔때마다 아래를 새로 정의해야 하나?
```python
net = torch.nn.Sequential(
    torch.nn.Embedding(num_embeddings=4,embedding_dim=20),
    torch.nn.Linear(in_features=20,out_features=20),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=20,out_features=4))
    #torch.nn.Softmax()
```

`-` 네트워크를 찍어내는 뭔가가 있음 좋지 않을까? 제가 만들어볼게요!

In [42]:
class BDA(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        u= self.linear1(self.embedding(X))
        v= self.relu(u)
        return self.linear2(v)

- 그냥 이렇게 간단해보이는 코드로 가능하다고? $\to$ 필요한 다른기능들은 Module에서 상속받았으므로 가능하다!

In [43]:
net2 = BDA(4) # 4는 dic의 size

In [44]:
net

Sequential(
  (0): Embedding(4, 20)
  (1): Linear(in_features=20, out_features=20, bias=True)
  (2): ReLU()
  (3): Linear(in_features=20, out_features=4, bias=True)
)

In [45]:
net2

BDA(
  (embedding): Embedding(4, 20)
  (linear1): Linear(in_features=20, out_features=20, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=20, out_features=4, bias=True)
)

- 오.. 

In [46]:
loss_fn = torch.nn.CrossEntropyLoss() 
optimizer2 = torch.optim.Adam(net2.parameters())

In [47]:
for i in range(1000):
    ## 1 
    yhat = net2(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward()
    ## 4 
    optimizer2.step()
    optimizer2.zero_grad()

In [48]:
net2(X)

tensor([[-4.6908,  5.4095, -4.1196, -4.1381],
        [-2.9237, -3.1811,  5.8576, -1.3189],
        [-4.6003, -4.9954,  3.0927,  3.0918],
        ...,
        [-2.9237, -3.1811,  5.8576, -1.3189],
        [-4.6003, -4.9954,  3.0927,  3.0918],
        [-4.6003, -4.9954,  3.0927,  3.0918]], grad_fn=<AddmmBackward0>)

In [49]:
net(X)

tensor([[-2.3892,  7.1199, -2.0946, -2.5325],
        [-3.2633, -3.4842,  6.0539, -3.6175],
        [-5.4399, -5.5046,  3.6418,  3.6417],
        ...,
        [-3.2633, -3.4842,  6.0539, -3.6175],
        [-5.4399, -5.5046,  3.6418,  3.6417],
        [-5.4399, -5.5046,  3.6418,  3.6417]], grad_fn=<AddmmBackward0>)

`-` net2도 잘 학습 되었다.

### 이전2단계

In [50]:
X = torch.tensor([nums[:498],nums[1:499]]).T
y = torch.tensor(nums[2:])

In [51]:
X[0],y[0] # he -> l 

(tensor([0, 1]), tensor(2))

In [52]:
X[1],y[1] # el -> l 

(tensor([1, 2]), tensor(2))

In [53]:
X[2],y[2] # ll -> o

(tensor([2, 2]), tensor(3))

`-` 아키텍처를 대충 스케치해보자. 

In [54]:
_e1 = torch.nn.Embedding(num_embeddings=4,embedding_dim=20)

In [55]:
X.shape, _e1(X).shape

(torch.Size([498, 2]), torch.Size([498, 2, 20]))

`-` 이전의 아키텍처는 아래와 같았음. 

```
torch.Size([499]) ## X with levels=4 
torch.Size([499, 20]) ## 임베딩 이후
torch.Size([499, 20]) ## 선형변환1 이후
torch.Size([499, 20]) ## 렐루 이후
torch.Size([499, 4]) ## 선형변환2 이후 
torch.Size([499, 4]) ## 소프트맥스 이후 = yhat 
```

`-` 벌써부터 꼬이는데?? 마지막 차원은 어쩌지? 합치나? 
- 합치는게 나쁜건 아닐지 몰라도 다른 대안이 있음 $\to$ 순환신경망의 발견 

`-` 순환신경망의 설계 

In [56]:
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        x1=X[:,0] # X의 첫번째 칼럼, y보다 2시점이전
        x2=X[:,1] # X의 두번째 칼럼, y보다 1시점이전 
        v=self.relu(self.linear1(self.embedding(x1))) # v: (498,20) 
        v2=self.relu(self.linear1(v+self.embedding(x2))) # 
        return self.linear2(v2)

`-` 결국 최종출력인 v2는 v와 x2이 담긴 함수이다. 그런데 v는 x1이 담긴 함수이다. 따라서 v2는 x2가 담겨있는 동시에 x1도 약하게 담겨있다 볼 수 있음. 

In [57]:
net3=BDA2(4)
net3

BDA2(
  (embedding): Embedding(4, 20)
  (linear1): Linear(in_features=20, out_features=20, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=20, out_features=4, bias=True)
)

In [58]:
net2

BDA(
  (embedding): Embedding(4, 20)
  (linear1): Linear(in_features=20, out_features=20, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=20, out_features=4, bias=True)
)

- 구조의 차이는 없지만 순전파 계산방식이 다르다! (그렇다면 역전파 계산방식도 다르겠지?..) 

`-` 다시 학습해보자. 

In [59]:
loss_fn = torch.nn.CrossEntropyLoss() # 사실손실함수는 계속 재활용해도 괜찮은데.. 어차피 같은거임 
optimizer3 = torch.optim.Adam(net3.parameters())

In [60]:
for i in range(1000):
    ## 1 
    yhat = net3(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward() 
    ## 4 
    optimizer3.step()
    optimizer3.zero_grad()

In [61]:
X[:5]

tensor([[0, 1],
        [1, 2],
        [2, 2],
        [2, 3],
        [3, 0]])

In [62]:
net3(X)[:5]

tensor([[-3.1893, -3.0848,  6.3955, -4.9327],
        [-2.8382, -9.0807,  7.9939, -0.0263],
        [-2.8636, -3.8070, -2.0860,  5.2985],
        [ 3.6494, -5.5746, -5.0601, -4.3493],
        [-6.7993,  6.9009, -4.0007, -2.8873]], grad_fn=<SliceBackward0>)

- he $\to$ l 
- el $\to$ l 
- ll $\to$ o 
- lo $\to$ h
- oh $\to$ e 

`-` 학습이 잘되었다. 

`-` 네트워크를 만드는 클래스를 조금 정리하자. 

```python 
### 원래 이런클래스였음 
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        x1=X[:,0] # X의 첫번째 칼럼, y보다 2시점이전
        x2=X[:,1] # X의 두번째 칼럼, y보다 1시점이전 
        v=self.relu(self.linear1(self.embedding(x1))) # v: (498,20) 
        v2=self.relu(self.linear1(v+self.embedding(x2))) # 
        return self.linear2(v2)

```

In [63]:
class BDA2(Module): ## 조금 정리한 클래스
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        v=0 
        for i in range(2):        
            v=self.relu(self.linear1(v+self.embedding(X[:,i]))) # 
        return self.linear2(v)

In [64]:
net3=BDA2(4)
loss_fn = torch.nn.CrossEntropyLoss() # 사실손실함수는 계속 재활용해도 괜찮은데.. 어차피 같은거임 
optimizer3 = torch.optim.Adam(net3.parameters())

In [65]:
for i in range(1000):
    ## 1 
    yhat = net3(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward() 
    ## 4 
    optimizer3.step()
    optimizer3.zero_grad()

In [66]:
X[:5]

tensor([[0, 1],
        [1, 2],
        [2, 2],
        [2, 3],
        [3, 0]])

In [67]:
net3(X)[:5]

tensor([[-2.4945, -2.6051,  6.3970, -3.7870],
        [-2.7388, -3.3491,  6.8680, -2.0323],
        [-0.6007, -2.0741, -0.7398,  7.7215],
        [ 7.6943, -2.7646, -1.9724, -0.7950],
        [-3.5048,  6.3504, -2.7557, -2.4894]], grad_fn=<SliceBackward0>)

- 동일한 결과가 나옴. 코드정리가 성공적이었음. 